In [ ]:
import boto3
from dotenv import load_dotenv
import os
import io
from PIL import Image
from tqdm import tqdm
import numpy as np
import pickle

np.set_printoptions(
    threshold=np.inf,
    linewidth=np.inf,
    precision=2,
    suppress=True
)

def transform_single_digit(img: Image):
    # Load and preprocess the image.
    img = img.convert('L')
    img = img.resize((28, 28), Image.Resampling.LANCZOS)

    # Done to allow the neural network to eventually process the individual symbols by inverting the colors.
    img_array = 255 - np.array(img)     
    img_array[img_array < 10] = 0

    return img_array

load_dotenv("../../backend/.env")

session = boto3.Session(
    aws_access_key_id=os.getenv('ACCESS_KEY'),
    aws_secret_access_key=os.getenv('SECRET_ACCESS_KEY'),
    aws_session_token=os.getenv('SESSION_TOKEN')
)

s3 = session.resource('s3')
path = "data/token/=/"
bucket = s3.Bucket("penman-lln")

objs = bucket.objects.filter(Prefix=path)

In [2]:
file_count = sum(1 for _ in objs)
images = np.zeros((file_count, 1, 28, 28))

for i, obj in tqdm(enumerate(objs)):
    if obj.key == path:
        continue

    curr_url = s3.Object("penman-lln", obj.key).get()["Body"].read()
    image = Image.open(io.BytesIO(curr_url))

    np_image = transform_single_digit(image)

    images[i] = np_image

219it [00:25,  8.73it/s]


In [4]:
# with open("../conv_neural_network/model/model.pkl", "rb") as file:
#     model = pickle.load(file)

print(images.shape)

(219, 1, 28, 28)


In [9]:
import torch

x = torch.tensor(images)

test_data = torch.tensor([]).new_full((x.size(0),), fill_value=4)

print(test_data.size())

torch.Size([219])
